<a href="https://colab.research.google.com/github/jarrodsb/ETAMU-binary-systems/blob/main/notebooks/RandomForestPrototype/Generate_Training_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Random Forest Classifier Toy Model for Classifying Planet Stability in Binary Systems

## Jarrod Bieber
## East Texas A&M University
## Summer 2026
---

## 1. Environment setup

In [1]:
!pip install rebound

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 904.0/904.0 kB 10.7 MB/s eta 0:00:00


In [2]:
import rebound
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as cm
import os

# Install scikit-learn
!pip install scikit-learn

# Import additional libraries
import pandas as pd
import seaborn as sns
import sklearn

# Install tqdm if not already installed for progress bar
try:
    from tqdm.notebook import tqdm
except ImportError:
    !pip install tqdm
    from tqdm.notebook import tqdm

## Data Generation
---

## 2. Define the parameter space and generate initial conditions

We will now generate `N` sets of initial conditions for S-type planetary systems. The parameters are sampled uniformly within specified ranges, as described in the problem statement.

In [ ]:
N = 2000 # Number of initial conditions to sample. Adjust this for faster runs.

# Define the ranges for parameter sampling
mu_range = [0.1, 0.9] # Binary mass ratio μ = m_B / (m_A + m_B)
e_bin_range = [0.0, 0.7] # Binary eccentricity
rho_range = [0.02, 0.5] # Planetary semimajor axis ratio ρ = a_p / a_bin (S-type range)
e_p_range = [0.0, 0.3] # Planetary eccentricity
inc_p_range_deg = [0.0, 40.0] # Planetary inclination relative to binary orbital plane in degrees
inc_p_range_rad = np.deg2rad(inc_p_range_deg) # Convert to radians for rebound
mean_anomaly_range_deg = [0.0, 360.0] # Initial planetary mean anomaly in degrees
mean_anomaly_range_rad = np.deg2rad(mean_anomaly_range_deg) # Convert to radians

# Store sampled parameters
sampled_params = {
    'mu': np.random.uniform(*mu_range, N),
    'e_bin': np.random.uniform(*e_bin_range, N),
    'rho': np.random.uniform(*rho_range, N),
    'e_p': np.random.uniform(*e_p_range, N),
    'inc_p': np.random.uniform(*inc_p_range_rad, N), # Storing in radians
    'mean_anomaly': np.random.uniform(*mean_anomaly_range_rad, N) # Storing in radians
}

# Total binary mass and binary semimajor axis (rebound units G=1)
M_total = 1.0 # total binary mass in solar masses
a_bin = 1.0   # binary semimajor axis in AU (rebound units)

print(f"Generated {N} sets of initial conditions.")

# Display the first 5 sampled parameters
# Converting to DataFrame for better display
df_sampled_params = pd.DataFrame(sampled_params)
df_sampled_params['inc_p_deg'] = np.rad2deg(df_sampled_params['inc_p'])
df_sampled_params['mean_anomaly_deg'] = np.rad2deg(df_sampled_params['mean_anomaly'])
display(df_sampled_params.head())

Generated 2000 sets of initial conditions.


,mu,e_bin,rho,e_p,inc_p,mean_anomaly,inc_p_deg,mean_anomaly_deg
0,0.277364,0.544496,0.353812,0.126107,0.203250,4.774376,11.645353,273.551573
1,0.123498,0.047661,0.461942,0.082044,0.561913,5.582934,32.195217,319.878564
2,0.769636,0.243600,0.334193,0.284526,0.190344,5.313474,10.905898,304.439607
3,0.734543,0.594010,0.280738,0.081287,0.423229,3.966059,24.249247,227.238464
4,0.370611,0.106712,0.356028,0.008383,0.067373,1.283788,3.860197,73.555615


## 3. Run short N-body integrations to extract summary features

For each sampled system, we will set up a `rebound` simulation with the two stars and the test-particle planet. We will integrate for a short baseline and compute several features that characterize the system's short-term dynamical behavior and physical properties. These features will then be used as inputs for our machine-learning model.

### Integrator Choice: IAS15 vs. WHFast

For these short integrations, we will use the **IAS15** integrator. IAS15 is a high-order, adaptive step-size integrator that is generally very accurate, especially for systems with close encounters or highly eccentric orbits, which might be present in our sampled parameter space. While **WHFast** is a symplectic integrator that conserves energy and angular momentum well over very long integration times, IAS15 is often preferred for shorter, more precise integrations where accurate tracking of orbital elements and chaos indicators like MEGNO is critical.

### Simulation Constants

In [ ]:
# Simulation constants
P_bin = 2 * np.pi # Binary orbital period for a_bin=1, M_total=1, G=1 (rebound units)
N_periods_short = 750 # Integration time for short simulation, in binary periods
T_short = N_periods_short * P_bin # Total integration time in rebound units

N_output_points = 200 # Number of points to sample during integration for features

# Note: Integration time (N_periods_short) and number of output points (N_output_points)
# are potential bottlenecks. Reduce these for faster first passes if needed.

print(f"Short integration time: {N_periods_short} binary periods ({T_short:.2f} rebound time units).")
print(f"Number of points for feature extraction: {N_output_points}")

Short integration time: 750 binary periods (4712.39 rebound time units).
Number of points for feature extraction: 200


### Features to be extracted:

1.  **`megno_median`**: The median value of the Mean Exponential Growth of Nearby Orbits (MEGNO) chaos indicator over the last 10% of the short integration. MEGNO quantifies the chaoticity of an orbit, with values close to 2 indicating regular motion and larger values indicating chaos.
2.  **`megno_std`**: The standard deviation of the MEGNO values over the last 80% of the integration. This helps to characterize the variability of the chaotic indicator, avoiding initial transients and providing a more robust measure of the system's long-term behavior.
3.  **`mu`**: The binary mass ratio, $m_B / (m_A + m_B)$. This is a fundamental system parameter.
4.  **`e_bin`**: The eccentricity of the binary orbit. This affects the strength of gravitational perturbations on the planet.
5.  **`rho`**: The ratio of the planetary semimajor axis to the binary semimajor axis, $a_p / a_{bin}$. This indicates how close the planet orbits to the binary.
6.  **`e_p_free`**: The free eccentricity of the planet. This is the component of the eccentricity that is not forced by the binary's gravitational influence. It represents the intrinsic eccentricity of the planet's orbit.
7.  **`e_p_forced`**: The forced eccentricity amplitude of the planet. This is the eccentricity induced on the planet's orbit by the time-averaged gravitational field of the binary. It's calculated from the magnitude of the time-averaged eccentricity vector.
8.  **`inc_p`**: The initial planetary inclination relative to the binary orbital plane. High inclinations can lead to Kozai-Lidov cycles, which can drive eccentricity to high values.
9.  **`a_p_std`**: The standard deviation of the planetary semimajor axis over the short integration, normalized by its initial value. This serves as a proxy for orbital energy diffusion; larger values indicate less stable orbits.
10. **`rho_crit_HW99`**: The analytical critical semimajor axis ratio derived from Holman & Wiegert (1999) for S-type planets, given the binary's mass ratio and eccentricity. This provides a theoretical baseline for the stability limit against which the ML model can be compared.

*Note: This set of features is a physically-adapted analogue of SPOCK's original features (Tamayo et al. 2020), tailored for S-type stability in binary systems where secular/Kozai perturbations are dominant, rather than planet-planet resonances.*

In [ ]:
def calculate_rho_crit_HW99(mu, e_bin):
    """
    Calculates the critical semimajor axis ratio (rho_crit = a_crit / a_bin)
    for an S-type planet based on the empirical fit from Holman & Wiegert (1999),
    Table 1, for S-type planets.

    Args:
        mu (float): Binary mass ratio (m_B / (m_A + m_B)).
        e_bin (float): Binary eccentricity.

    Returns:
        float: The critical semimajor axis ratio (a_crit / a_bin).
    """
    # Empirical fit from Holman & Wiegert (1999), Table 1 for S-type planets
    # a_crit / a_bin = C1 + C2*mu + C3*e_bin + C4*mu*e_bin + C5*e_bin^2 + C6*mu*e_bin^2
    # Note: The original paper sometimes presents different fits. This one is commonly cited.
    # Using coefficients from the paper directly (Table 1, S-type stability limit)
    rho_crit = 0.464 - 0.380 * mu - 0.631 * e_bin + 0.150 * mu * e_bin + 0.198 * e_bin**2 + 0.088 * mu * e_bin**2
    return rho_crit

def run_short_simulation_and_extract_features(params):
    """
    Sets up and runs a short rebound simulation for a given set of parameters,
    then extracts the specified features.

    Args:
        params (dict): A dictionary containing the parameters for one system:
                       'mu', 'e_bin', 'rho', 'e_p', 'inc_p', 'mean_anomaly'.

    Returns:
        dict: A dictionary containing the extracted features.
    """
    mu = params['mu']
    e_bin = params['e_bin']
    rho = params['rho']
    e_p = params['e_p']
    inc_p = params['inc_p'] # Already in radians
    M_p = params.get('M_p', 0.) # Planet mass (default to 0 for test particle)
    mean_anomaly_p = params['mean_anomaly'] # Already in radians

    # Calculate stellar masses
    m_B = mu * M_total
    m_A = M_total - m_B

    # Planetary semimajor axis
    a_p = rho * a_bin

    sim = rebound.Simulation()
    sim.integrator = "ias15"
    sim.G = 1.0

    # Add the primary star (star A) at the origin.
    # This particle will be sim.particles[0]
    sim.add(m=m_A)

    # Add the secondary star (star B) orbiting the primary star (sim.particles[0]).
    # The parameters (a, e, inc, etc.) define the relative orbit of B around A.
    # The binary's orbital plane is set as the xy-plane (inc=0 relative to primary).
    # This particle will be sim.particles[1]
    sim.add(m=m_B, primary=sim.particles[0], a=a_bin, e=e_bin, inc=0, Omega=0, omega=0, M=0)

    # Add the planet (test particle) orbiting the primary star (sim.particles[0]).
    # Its inclination (inc_p) is relative to the binary orbital plane (which is currently the xy-plane).
    # Its mean anomaly (mean_anomaly_p) is given.
    # This particle will be sim.particles[2]
    sim.add(m=M_p, primary=sim.particles[0], a=a_p, e=e_p, inc=inc_p, Omega=0, omega=0, M=mean_anomaly_p)

    # Move the entire system to its center of momentum frame.
    # This step is crucial for accurate simulation as it places the origin at the barycenter,
    # which ensures the system's total momentum is zero.
    sim.move_to_com()

    sim.dt = sim.particles[1].P/20. # Set initial timestep for IAS15, will be adapted
    # If using WHFast, typically dt ~ P_min / (10-20), where P_min is the shortest period
    # sim.integrator = "whfast"
    # sim.dt = sim.particles[2].P / 20. # Example for WHFast, needs careful choice

    # Enable MEGNO calculation
    # sim.track_energy_momentum = True # Required for MEGNO - This line is removed as it's no longer needed or supported.
    sim.init_megno() # Initialize MEGNO

    # Store orbital elements and MEGNO values
    times = np.linspace(0, T_short, N_output_points)
    megnos = []
    a_ps = []
    e_vec_x = []
    e_vec_y = []

    try:
        for i, time in enumerate(times):
            sim.integrate(time)
            megnos.append(sim.megno())
            a_ps.append(sim.particles[2].a)
            e_vec_x.append(sim.particles[2].e * np.cos(sim.particles[2].pomega))
            e_vec_y.append(sim.particles[2].e * np.sin(sim.particles[2].pomega))

    except rebound.Escape as error:
        # If the planet escapes, label it as unstable and stop integration
        # We will handle the stability label in a later step (long integration)
        # For now, just return NaNs for features if integration fails early
        print(f"Warning: Planet escaped during short integration. Error: {error}")
        return {
            'mu': mu,
            'e_bin': e_bin,
            'rho': rho,
            'e_p': e_p,
            'inc_p': inc_p,
            'megno_median': np.nan,
            'megno_std': np.nan,
            'e_p_free': np.nan,
            'e_p_forced': np.nan,
            'a_p_std': np.nan,
            'rho_crit_HW99': calculate_rho_crit_HW99(mu, e_bin)
        }
    except Exception as e:
        print(f"An error occurred during integration: {e}")
        return {
            'mu': mu,
            'e_bin': e_bin,
            'rho': rho,
            'e_p': e_p,
            'inc_p': inc_p,
            'megno_median': np.nan,
            'megno_std': np.nan,
            'e_p_free': np.nan,
            'e_p_forced': np.nan,
            'a_p_std': np.nan,
            'rho_crit_HW99': calculate_rho_crit_HW99(mu, e_bin)
        }

    # Convert lists to numpy arrays for calculations
    megnos = np.array(megnos)
    a_ps = np.array(a_ps)
    e_vec_x = np.array(e_vec_x)
    e_vec_y = np.array(e_vec_y)

    # 1. megno_median: median MEGNO chaos indicator over the last 10% of the integration
    megno_median = np.nanmedian(megnos[int(0.9 * N_output_points):])

    # 2. megno_std: standard deviation of MEGNO over the last 80% (avoid initial transients)
    megno_std = np.nanstd(megnos[int(0.2 * N_output_points):])

    # 6. e_p_free and 7. e_p_forced
    # Time-averaged eccentricity vector
    if len(e_vec_x) > 0:
        mean_e_vec_x = np.nanmean(e_vec_x)
        mean_e_vec_y = np.nanmean(e_vec_y)

        e_p_forced = np.sqrt(mean_e_vec_x**2 + mean_e_vec_y**2)

        # Free eccentricity is the magnitude of the difference vector
        e_p_free_components_x = e_vec_x - mean_e_vec_x
        e_p_free_components_y = e_vec_y - mean_e_vec_y
        e_p_free_instantaneous = np.sqrt(e_p_free_components_x**2 + e_p_free_components_y**2)
        e_p_free = np.nanmean(e_p_free_instantaneous) # Averaging the magnitude of the free component
    else:
        e_p_forced = np.nan
        e_p_free = np.nan

    # 9. a_p_std: standard deviation of the planetary semimajor axis, normalized by initial a_p
    initial_a_p = a_p # This is the initial a_p from the input parameters
    a_p_std = np.nanstd(a_ps) / initial_a_p if initial_a_p != 0 else np.nan

    # 10. rho_crit_HW99
    rho_crit_HW99_val = calculate_rho_crit_HW99(mu, e_bin)

    return {
        'mu': mu,
        'e_bin': e_bin,
        'rho': rho,
        'e_p': e_p,
        'inc_p': np.rad2deg(inc_p), # Store in degrees for consistency with input range description
        'megno_median': megno_median,
        'megno_std': megno_std,
        'e_p_free': e_p_free,
        'e_p_forced': e_p_forced,
        'a_p_std': a_p_std,
        'rho_crit_HW99': rho_crit_HW99_val
    }

### Running simulations and extracting features

Now we iterate through all `N` sampled initial conditions, run the short `rebound` simulation for each, and collect the features. This step can be computationally intensive, especially for larger `N` or longer `N_periods_short`.

To track progress, we'll use `tqdm` if available (a progress bar library).

In [ ]:
features_list = []

# Convert sampled_params dictionary to a list of dictionaries for easier iteration
list_of_params = []
for i in range(N):
    list_of_params.append({
        'mu': sampled_params['mu'][i],
        'e_bin': sampled_params['e_bin'][i],
        'rho': sampled_params['rho'][i],
        'e_p': sampled_params['e_p'][i],
        'inc_p': sampled_params['inc_p'][i],
        'mean_anomaly': sampled_params['mean_anomaly'][i]
    })

# Reduce the number of simulations for faster runtime during development/testing
N_to_run = 100 # You can change this number. Set to N to run all original simulations.
list_of_params_subset = list_of_params[:N_to_run]

print(f"Starting {N_to_run} short simulations... This may take a while.")

# Loop through each set of initial conditions
for i, params in enumerate(tqdm(list_of_params_subset, desc=f"Running {N_to_run} short simulations")):
    features = run_short_simulation_and_extract_features(params)
    features_list.append(features)

print("Short simulations complete.")

Starting 100 short simulations... This may take a while.


Running 100 short simulations:   0%|          | 0/100 [00:00<?, ?it/s]

Short simulations complete.


## 4. Determine the stability label

To determine the long-term stability of each system, we will run a longer integration for a representative subsample (or all systems if compute allows). The planet is considered `stable = 1` if it survives the integration without ejection or collision, and `unstable = 0` otherwise. `rebound`'s `exit_min_distance` and `escape` functionalities are crucial here to save computational resources by terminating unstable systems early.

In [ ]:
# Simulation constants for long integration
# Note: This is the primary bottleneck for computation. Adjust N_periods_long accordingly.
N_periods_long = 1e4 # Integration time for long simulation, in binary periods (10^4)
T_long = N_periods_long * P_bin # Total integration time in rebound units

# Define thresholds for instability detection
# If a particle gets too close to a star, or escapes too far, we consider it unstable.
MIN_DIST_FACTOR = 0.5 # Planet cannot get closer than 0.5*a_bin to primary (or secondary)
ESCAPE_DIST_FACTOR = 2.0 # Planet cannot go further than 2.0*a_bin from primary

print(f"Long integration time: {N_periods_long} binary periods ({T_long:.2f} rebound time units).")
print(f"Minimum distance factor (a_bin): {MIN_DIST_FACTOR}")
print(f"Escape distance factor (a_bin): {ESCAPE_DIST_FACTOR}")

def run_long_simulation_and_label_stability(params, a_p_initial):
    """
    Sets up and runs a long rebound simulation for a given set of parameters
    and determines the stability label.

    Args:
        params (dict): A dictionary containing the initial parameters for the system.
        a_p_initial (float): The initial planetary semimajor axis.

    Returns:
        int: 1 for stable, 0 for unstable.
    """
    mu = params['mu']
    e_bin = params['e_bin']
    rho = params['rho']
    e_p = params['e_p']
    inc_p = params['inc_p'] # Already in radians
    M_p = params.get('M_p', 0.) # Planet mass (default to 0 for test particle)
    mean_anomaly_p = params['mean_anomaly'] # Already in radians

    # Calculate stellar masses
    m_B = mu * M_total
    m_A = M_total - m_B

    # Planetary semimajor axis
    a_p = rho * a_bin

    sim = rebound.Simulation()
    sim.integrator = "ias15"
    sim.G = 1.0

    sim.add(m=m_A)
    sim.add(m=m_B, primary=sim.particles[0], a=a_bin, e=e_bin, inc=0, Omega=0, omega=0, M=0)
    sim.add(m=M_p, primary=sim.particles[0], a=a_p, e=e_p, inc=inc_p, Omega=0, omega=0, M=mean_anomaly_p)

    sim.move_to_com()
    sim.dt = sim.particles[1].P/20. # Initial timestep for IAS15

    # Set up exit conditions for instability
    # Planet index is 2
    # Exit if planet gets too close to either star
    sim.exit_min_distance = MIN_DIST_FACTOR * a_bin # Minimum distance from barycenter for planet

    # Exit if planet escapes (e.g., goes beyond 2*a_bin from the primary star)
    # We set this relative to the barycenter for simplicity, or relative to the central body
    # rebound.simulation.exit_distance sets a distance from the barycenter for all particles
    # To check specifically for the planet relative to the primary, we need to add a custom exit condition.

    # For simplicity, let's just use the sim.exit_min_distance as the closest approach to any other particle
    # and let the rebound.Escape exception handle very large distances implicitly.
    # More refined escape conditions can be added if needed, for instance, checking particle-specific distances.

    try:
        # Integrate until T_long or until an exit condition is met
        sim.integrate(T_long)
        # If integration completes, system is stable
        stability_label = 1
    except rebound.Escape:
        # If an escape occurs, system is unstable
        stability_label = 0
    except Exception as e:
        # Catch any other integration errors as unstable
        print(f"An unexpected error occurred during long integration: {e}")
        stability_label = 0

    return stability_label


Long integration time: 10000.0 binary periods (62831.85 rebound time units).
Minimum distance factor (a_bin): 0.5
Escape distance factor (a_bin): 2.0


### Running long integrations and assigning stability labels

This step is the most computationally intensive. For the initial run, we will perform long integrations for all `N` systems. If this is too slow, `N` should be reduced, or a subsample can be used instead.

In [ ]:
stability_labels = []

print(f"Starting {N_to_run} long simulations to determine stability...")

for i, params in enumerate(tqdm(list_of_params_subset, desc="Running long simulations")):
    # Pass the initial planetary semimajor axis for consistency with the feature extraction if needed
    # For stability, we only need the label, not the raw a_p.
    initial_a_p_for_stability_check = params['rho'] * a_bin
    label = run_long_simulation_and_label_stability(params, initial_a_p_for_stability_check)
    stability_labels.append(label)

print("Long simulations complete.")

Starting 100 long simulations to determine stability...


Running long simulations:   0%|          | 0/100 [00:00<?, ?it/s]

An unexpected error occurred during long integration: Two particles had a close encounter (d<exit_min_distance).
An unexpected error occurred during long integration: Two particles had a close encounter (d<exit_min_distance).
An unexpected error occurred during long integration: Two particles had a close encounter (d<exit_min_distance).
An unexpected error occurred during long integration: Two particles had a close encounter (d<exit_min_distance).
An unexpected error occurred during long integration: Two particles had a close encounter (d<exit_min_distance).
An unexpected error occurred during long integration: Two particles had a close encounter (d<exit_min_distance).
An unexpected error occurred during long integration: Two particles had a close encounter (d<exit_min_distance).
An unexpected error occurred during long integration: Two particles had a close encounter (d<exit_min_distance).
An unexpected error occurred during long integration: Two particles had a close encounter (d<exi

## 8. Generate full dataset with multiprocessing and checkpointing

To generate a larger dataset efficiently, we will compile the data generation, simulation, and feature extraction steps into a single Python script. This script will leverage `multiprocessing` to run simulations in parallel, `tqdm` for progress visualization, and implement checkpointing to save results incrementally to a CSV file. This setup allows for unattended execution, robust error handling, and efficient resource utilization, especially for long-running simulations.

The script will be saved as `generate_dataset_v2.py` and can be run from the command line, for example:

```bash
N_SYSTEMS=5000 python3 generate_dataset_v2.py
```

This command will generate 5000 systems and save their features and stability labels to `s_type_stability_data.csv`.

In [3]:
%%writefile generate_dataset_v2.py

import rebound
import numpy as np
import pandas as pd
import os
import csv
import multiprocessing
import sys
import time # Import time module

# Check if running in a notebook or as a script
# This helps select the correct tqdm import for better display
if 'ipykernel' in sys.modules:
    from tqdm.notebook import tqdm
else:
    from tqdm import tqdm

# Define the ranges for parameter sampling
mu_range = [0.1, 0.9] # Binary mass ratio μ = m_B / (m_A + m_B)
e_bin_range = [0.0, 0.7] # Binary eccentricity
rho_range = [0.02, 0.5] # Planetary semimajor axis ratio ρ = a_p / a_bin (S-type range)
e_p_range = [0.0, 0.3] # Planetary eccentricity
inc_p_range_deg = [0.0, 40.0] # Planetary inclination relative to binary orbital plane in degrees
inc_p_range_rad = np.deg2rad(inc_p_range_deg) # Convert to radians for rebound
mean_anomaly_range_deg = [0.0, 360.0] # Initial planetary mean anomaly in degrees
mean_anomaly_range_rad = np.deg2rad(mean_anomaly_range_deg) # Convert to radians

# Total binary mass and binary semimajor axis (rebound units G=1)
M_total = 1.0 # total binary mass in solar masses
a_bin = 1.0   # binary semimajor axis in AU (rebound units)

# Simulation constants
P_bin = 2 * np.pi # Binary orbital period for a_bin=1, M_total=1, G=1 (rebound units)
N_periods_short = 750 # Integration time for short simulation, in binary periods
T_short = N_periods_short * P_bin # Total integration time in rebound units
N_output_points = 200 # Number of points to sample during integration for features

# Simulation constants for long integration
N_periods_long = 1e4 # Integration time for long simulation, in binary periods (10^4)
T_long = N_periods_long * P_bin # Total integration time in rebound units

# Define thresholds for instability detection
MIN_DIST_FACTOR = 0.5 # Planet cannot get closer than 0.5*a_bin to primary (or secondary)
ESCAPE_DIST_FACTOR = 2.0 # Planet cannot go further than 2.0*a_bin from primary

def calculate_rho_crit_HW99(mu, e_bin):
    """
    Calculates the critical semimajor axis ratio (rho_crit = a_crit / a_bin)
    for an S-type planet based on the empirical fit from Holman & Wiegert (1999),
    Table 1, for S-type planets.

    Args:
        mu (float): Binary mass ratio (m_B / (m_A + m_B)).
        e_bin (float): Binary eccentricity.

    Returns:
        float: The critical semimajor axis ratio (a_crit / a_bin).
    """
    rho_crit = 0.464 - 0.380 * mu - 0.631 * e_bin + 0.150 * mu * e_bin + 0.198 * e_bin**2 + 0.088 * mu * e_bin**2
    return rho_crit

def run_short_simulation_and_extract_features(params):
    """
    Sets up and runs a short rebound simulation for a given set of parameters,
    then extracts the specified features.

    Args:
        params (dict): A dictionary containing the parameters for one system:
                       'mu', 'e_bin', 'rho', 'e_p', 'inc_p', 'mean_anomaly', 'id'.

    Returns:
        tuple: (dict: extracted features, str: reason for instability or 'ok'). Returns NaNs for features
              if an escape or error occurs.
    """
    mu = params['mu']
    e_bin = params['e_bin']
    rho = params['rho']
    e_p = params['e_p']
    inc_p = params['inc_p'] # Already in radians
    M_p = params.get('M_p', 0.) # Planet mass (default to 0 for test particle)
    mean_anomaly_p = params['mean_anomaly'] # Already in radians
    system_id = params.get('id', 'N/A')

    # Calculate stellar masses
    m_B = mu * M_total
    m_A = M_total - m_B

    # Planetary semimajor axis
    a_p = rho * a_bin

    sim = rebound.Simulation()
    sim.integrator = "ias15"
    sim.G = 1.0

    sim.add(m=m_A)
    sim.add(m=m_B, primary=sim.particles[0], a=a_bin, e=e_bin, inc=0, Omega=0, omega=0, M=0)
    sim.add(m=M_p, primary=sim.particles[0], a=a_p, e=e_p, inc=inc_p, Omega=0, omega=0, M=mean_anomaly_p)

    sim.move_to_com()
    sim.dt = sim.particles[1].P/20.
    sim.init_megno() # Initialize MEGNO

    times = np.linspace(0, T_short, N_output_points)
    megnos = []
    a_ps = []
    e_vec_x = []
    e_vec_y = []

    features = {
        'mu': mu,
        'e_bin': e_bin,
        'rho': rho,
        'e_p': e_p,
        'inc_p': np.rad2deg(inc_p),
        'megno_median': np.nan,
        'megno_std': np.nan,
        'e_p_free': np.nan,
        'e_p_forced': np.nan,
        'a_p_std': np.nan,
        'rho_crit_HW99': calculate_rho_crit_HW99(mu, e_bin),
        'stable': 0 # Default to unstable, will be set to 1 later if stable
    }
    stability_reason = 'ok'

    try:
        for i, time in enumerate(times):
            sim.integrate(time)
            megnos.append(sim.megno())
            a_ps.append(sim.particles[2].a)
            e_vec_x.append(sim.particles[2].e * np.cos(sim.particles[2].pomega))
            e_vec_y.append(sim.particles[2].e * np.sin(sim.particles[2].pomega))

    except rebound.Escape as error:
        print(f"System {system_id} had an Escape during short integration.")
        stability_reason = 'escape'
        return features, stability_reason
    except rebound.Encounter as error:
        print(f"System {system_id} had an Encounter during short integration.")
        stability_reason = 'encounter'
        return features, stability_reason
    except Exception as e:
        sys.stderr.write(f"System {system_id} had an unexpected error during short integration: {e}\n")
        stability_reason = 'short_error'
        return features, stability_reason

    megnos = np.array(megnos)
    a_ps = np.array(a_ps)
    e_vec_x = np.array(e_vec_x)
    e_vec_y = np.array(e_vec_y)

    features['megno_median'] = np.nanmedian(megnos[int(0.9 * N_output_points):]) if len(megnos) > 0 else np.nan
    features['megno_std'] = np.nanstd(megnos[int(0.2 * N_output_points):]) if len(megnos) > 0 else np.nan

    if len(e_vec_x) > 0:
        mean_e_vec_x = np.nanmean(e_vec_x)
        mean_e_vec_y = np.nanmean(e_vec_y)
        features['e_p_forced'] = np.sqrt(mean_e_vec_x**2 + mean_e_vec_y**2)
        e_p_free_components_x = e_vec_x - mean_e_vec_x
        e_p_free_components_y = e_vec_y - mean_e_vec_y
        e_p_free_instantaneous = np.sqrt(e_p_free_components_x**2 + e_p_free_components_y**2)
        features['e_p_free'] = np.nanmean(e_p_free_instantaneous)
    else:
        features['e_p_forced'] = np.nan
        features['e_p_free'] = np.nan

    initial_a_p = a_p # This is the initial a_p from the input parameters
    features['a_p_std'] = np.nanstd(a_ps) / initial_a_p if initial_a_p != 0 and len(a_ps) > 0 else np.nan

    features['rho_crit_HW99'] = calculate_rho_crit_HW99(mu, e_bin)
    features['stable'] = -1 # Placeholder, will be updated by long integration if short integration was successful

    return features, stability_reason

def run_long_simulation_and_label_stability(initial_params):
    """
    Sets up and runs a long rebound simulation for a given set of parameters
    and determines the stability label. Catches rebound.Escape and rebound.Encounter.

    Args:
        initial_params (dict): A dictionary containing the initial parameters for the system.
                                Expected to contain 'mu', 'e_bin', 'rho', 'e_p', 'inc_p' (radians),
                                'mean_anomaly', 'id'.

    Returns:
        tuple: (int: 1 for stable, 0 for unstable, str: reason for instability or 'stable').
    """
    mu = initial_params['mu']
    e_bin = initial_params['e_bin']
    rho = initial_params['rho']
    e_p = initial_params['e_p']
    inc_p = initial_params['inc_p'] # Already in radians
    M_p = initial_params.get('M_p', 0.)
    mean_anomaly_p = initial_params['mean_anomaly'] # Already in radians
    system_id = initial_params.get('id', 'N/A')

    m_B = mu * M_total
    m_A = M_total - m_B
    a_p = rho * a_bin

    sim = rebound.Simulation()
    sim.integrator = "ias15"
    sim.G = 1.0

    sim.add(m=m_A)
    sim.add(m=m_B, primary=sim.particles[0], a=a_bin, e=e_bin, inc=0, Omega=0, omega=0, M=0)
    sim.add(m=M_p, primary=sim.particles[0], a=a_p, e=e_p, inc=inc_p, Omega=0, omega=0, M=mean_anomaly_p)

    sim.move_to_com()
    sim.dt = sim.particles[1].P/20.

    # Set up exit conditions for instability
    sim.exit_min_distance = MIN_DIST_FACTOR * a_bin # Minimum distance for planet to any other body (barycenter distance)
    # The rebound.Escape exception handles particles going too far.

    stability_label = 0
    stability_reason = 'long_error' # Default to error until proven otherwise

    try:
        sim.integrate(T_long)
        stability_label = 1 # If integration completes, system is stable
        stability_reason = 'stable'
    except rebound.Escape as error:
        print(f"System {system_id} had an Escape during long integration.")
        stability_reason = 'escape'
    except rebound.Encounter as error:
        print(f"System {system_id} had an Encounter during long integration.")
        stability_reason = 'encounter'
    except Exception as e:
        sys.stderr.write(f"System {system_id} had an unexpected error during long integration: {e}\n")
        stability_reason = 'long_error'

    return stability_label, stability_reason

def worker_simulate_system(system_params):
    """
    Worker function to run both short and long simulations for a single system.
    """
    # First, run short simulation and extract features
    features, short_reason = run_short_simulation_and_extract_features(system_params)

    # If the short simulation already indicated instability (e.g., due to escape/encounter/error),
    # don't run long integration and mark as unstable with the reason from the short sim.
    if short_reason != 'ok':
        features['stable'] = 0
        features['reason'] = short_reason
    else:
        # Then, run long simulation to determine stability label
        stability_label, long_reason = run_long_simulation_and_label_stability(system_params)
        features['stable'] = stability_label
        features['reason'] = long_reason

    return features

def main():
    start_time = time.time() # Record start time
    N_SYSTEMS = int(os.getenv('N_SYSTEMS', '100')) # Default to 100 for quick testing
    OUTPUT_FILENAME = f"s_type_stability_data_{N_SYSTEMS}.csv" # Updated filename
    CHECKPOINT_INTERVAL = 500
    N_WORKERS = int(os.getenv('N_WORKERS', '11')) # Default to 11 workers

    print(f"Generating {N_SYSTEMS} systems using {N_WORKERS} workers.")
    print(f"Output will be saved to {OUTPUT_FILENAME} with checkpointing every {CHECKPOINT_INTERVAL} systems.")

    # Generate all initial conditions once
    sampled_params = {
        'mu': np.random.uniform(*mu_range, N_SYSTEMS),
        'e_bin': np.random.uniform(*e_bin_range, N_SYSTEMS),
        'rho': np.random.uniform(*rho_range, N_SYSTEMS),
        'e_p': np.random.uniform(*e_p_range, N_SYSTEMS),
        'inc_p': np.random.uniform(*inc_p_range_rad, N_SYSTEMS),
        'mean_anomaly': np.random.uniform(*mean_anomaly_range_rad, N_SYSTEMS)
    }

    list_of_params_for_workers = []
    for i in range(N_SYSTEMS):
        list_of_params_for_workers.append({
            'id': i, # Add system ID
            'mu': sampled_params['mu'][i],
            'e_bin': sampled_params['e_bin'][i],
            'rho': sampled_params['rho'][i],
            'e_p': sampled_params['e_p'][i],
            'inc_p': sampled_params['inc_p'][i],
            'mean_anomaly': sampled_params['mean_anomaly'][i]
        })

    # Prepare CSV file header if it doesn't exist
    # We need to run one dummy simulation to get all feature keys for the header
    dummy_features = worker_simulate_system(list_of_params_for_workers[0])
    # The 'id' key was temporary for error reporting and should not be a feature in the CSV
    if 'id' in dummy_features:
        del dummy_features['id']
    # The 'reason' key is for internal tracking, not to be saved in CSV
    if 'reason' in dummy_features:
        del dummy_features['reason']
    fieldnames = list(dummy_features.keys())

    file_exists = os.path.isfile(OUTPUT_FILENAME)
    if not file_exists:
        with open(OUTPUT_FILENAME, 'w', newline='') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()

    # Counters for final metrics
    escape_count = 0
    encounter_count = 0
    stable_count = 0
    other_unstable_count = 0 # For short_error, long_error

    # Use multiprocessing Pool
    completed_systems = 0
    with multiprocessing.Pool(processes=N_WORKERS) as pool:
        # Use imap_unordered to process results as they come in
        for i, result_features in enumerate(tqdm(
            pool.imap_unordered(worker_simulate_system, list_of_params_for_workers),
            total=N_SYSTEMS, desc="Simulating systems")
        ):
            # Update counts based on the 'reason'
            if result_features['stable'] == 1:
                stable_count += 1
            elif result_features['reason'] == 'escape':
                escape_count += 1
            elif result_features['reason'] == 'encounter':
                encounter_count += 1
            else: # All other unstable reasons (short_error, long_error)
                other_unstable_count += 1

            # Ensure the 'id' and 'reason' keys are not written to the CSV
            if 'id' in result_features:
                del result_features['id']
            if 'reason' in result_features:
                del result_features['reason']

            with open(OUTPUT_FILENAME, 'a', newline='') as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writerow(result_features)

            completed_systems += 1
            if completed_systems % CHECKPOINT_INTERVAL == 0:
                sys.stdout.write(f"\nCheckpoint: {completed_systems}/{N_SYSTEMS} systems processed and saved.\n")

    end_time = time.time() # Record end time
    elapsed_time = end_time - start_time

    print(f"\nFinished generating {N_SYSTEMS} systems. Data saved to {OUTPUT_FILENAME}.")
    print(f"\nSummary of results:")
    print(f"  {escape_count} systems had an Escape.")
    print(f"  {encounter_count} systems had an Encounter.")
    print(f"  {stable_count} systems were stable.")
    print(f"  {other_unstable_count} systems were unstable due to other errors (short/long integration). ")
    print(f"Total elapsed time: {elapsed_time:.2f} seconds.")

    # Optional: Display final statistics (only if run in Colab or if script wants to print summary)
    if 'ipykernel' in sys.modules:
        df_final = pd.read_csv(OUTPUT_FILENAME)
        print("\nFinal DataFrame head:")
        display(df_final.head())
        print(f"Total stable systems: {df_final['stable'].sum()} out of {len(df_final)}")
        print(f"Total unstable systems: {len(df_final) - df_final['stable'].sum()} out of {len(df_final)}")


if __name__ == '__main__':
    main()

Writing generate_dataset_v2.py


## 9. Generate full multi-planet s-type binary dataset

### Features to be extracted for Two-Planet Systems:

**A. System-wide / Binary-driven Parameters (Common for both planets):**

1.  **`mu`**: The binary mass ratio, $m_B / (m_A + m_B)$. A fundamental system parameter describing the mass distribution within the binary.
2.  **`e_bin`**: The eccentricity of the binary orbit. Influences the strength and periodicity of gravitational perturbations on the planets.
3.  **`rho_crit_HW99_p1`**: The analytical critical semimajor axis ratio derived from Holman & Wiegert (1999), Table 1, for S-type planets. This is calculated using the binary's properties and serves as an empirical stability boundary for the inner planet.

**B. Planet-Specific Features (extracted for each planet, suffixed `_p1` for inner, `_p2` for outer):**

4.  **`rho`**: The ratio of the planetary semimajor axis to the binary semimajor axis, $a_p / a_{bin}$. Indicates how tightly the planet orbits within the binary system.
5.  **`e_p_free`**: The free eccentricity of the planet. Represents the component of eccentricity not forced by the binary's time-averaged gravitational field. It reflects the planet's intrinsic orbital shape.
6.  **`e_p_forced`**: The forced eccentricity amplitude of the planet. This is the eccentricity induced on the planet's orbit by the quasi-static gravitational field of the binary. It's calculated from the magnitude of the time-averaged eccentricity vector.
7.  **`inc_p_deg`**: The initial planetary inclination relative to the binary orbital plane (in degrees). High inclinations can lead to Kozai-Lidov cycles, potentially driving eccentricity variations.
8.  **`megno_median`**: The median value of the Mean Exponential Growth of Nearby Orbits (MEGNO) chaos indicator over the last 10% of the short integration. Values near 2 suggest regular motion, larger values indicate chaos.
9.  **`megno_std`**: The standard deviation of the MEGNO values over the last 80% of the integration. Characterizes the variability of the chaotic indicator, providing a robust measure by avoiding initial transients.
10. **`a_p_std`**: The standard deviation of the planetary semimajor axis over the short integration, normalized by its initial value. A proxy for orbital energy diffusion; larger values suggest less stable orbits and potential close encounters.

**C. Planet-Planet Interaction Features:**

11. **`period_ratio_p2_p1`**: The ratio of the initial orbital period of planet 2 to planet 1 ($P_2/P_1$). Values close to integer or half-integer ratios indicate potential mean-motion resonances.
12. **`initial_separation_au`**: The initial physical separation between planet 2 and planet 1, in astronomical units (since $a_{bin}=1$). ($a_{p2} - a_{p1}$).
13. **`init_mutual_inc_deg`**: The initial mutual inclination between the orbital planes of planet 1 and planet 2 (in degrees). This is the absolute difference in their inclinations assuming aligned nodes (since initial Omega is 0).
14. **`mmr_flag_2_1`**: A binary flag (1 if true, 0 if false) indicating if the initial period ratio `period_ratio_p2_p1` is approximately 2:1 (e.g., within 5% of 2.0). This signifies a potential 2:1 mean-motion resonance.
15. **`mmr_flag_3_2`**: A binary flag (1 if true, 0 if false) indicating if the initial period ratio `period_ratio_p2_p1` is approximately 3:2 (e.g., within 5% of 1.5). This signifies a potential 3:2 mean-motion resonance.

In [ ]:
%%writefile generate_dataset_v2_two_planets.py

import rebound
import numpy as np
import pandas as pd
import os
import csv
import multiprocessing
import sys
import time

# Check if running in a notebook or as a script
# This helps select the correct tqdm import for better display
if 'ipykernel' in sys.modules:
    from tqdm.notebook import tqdm
else:
    from tqdm import tqdm

# Define the ranges for parameter sampling for two planets
mu_range = [0.1, 0.9] # Binary mass ratio μ = m_B / (m_A + m_B)
e_bin_range = [0.0, 0.7] # Binary eccentricity

# Planetary semimajor axis ratio ρ = a_p / a_bin (S-type range)
# For two planets, ensure a minimum separation
rho_min_global = 0.02
rho_max_global = 0.5
min_period_ratio_between_planets = 1.1 # Ensures P2/P1 > 1.1, avoids too close initial conditions

e_p_range = [0.0, 0.3] # Planetary eccentricity
inc_p_range_deg = [0.0, 40.0] # Planetary inclination relative to binary orbital plane in degrees
inc_p_range_rad = np.deg2rad(inc_p_range_deg) # Convert to radians for rebound
mean_anomaly_range_deg = [0.0, 360.0] # Initial planetary mean anomaly in degrees
mean_anomaly_range_rad = np.deg2rad(mean_anomaly_range_deg) # Convert to radians

# Total binary mass and binary semimajor axis (rebound units G=1)
M_total = 1.0 # total binary mass in solar masses
a_bin = 1.0   # binary semimajor axis in AU (rebound units)

# Simulation constants
P_bin = 2 * np.pi # Binary orbital period for a_bin=1, M_total=1, G=1 (rebound units)
N_periods_short = 750 # Integration time for short simulation, in binary periods
T_short = N_periods_short * P_bin # Total integration time in rebound units
N_output_points = 200 # Number of points to sample during integration for features

# Simulation constants for long integration
N_periods_long = 1e4 # Integration time for long simulation, in binary periods (10^4)
T_long = N_periods_long * P_bin # Total integration time in rebound units

# Define thresholds for instability detection
MIN_DIST_FACTOR = 0.5 # Planet cannot get closer than 0.5*a_bin to primary (or secondary)
ESCAPE_DIST_FACTOR = 2.0 # Planet cannot go further than 2.0*a_bin from primary

# For mutual Hill radius calculation (even for test particles, need a reference mass)
M_p_ref_for_features = 1e-5 * M_total # A small, non-zero mass for feature calculations

def calculate_rho_crit_HW99(mu, e_bin):
    """
    Calculates the critical semimajor axis ratio (rho_crit = a_crit / a_bin)
    for an S-type planet based on the empirical fit from Holman & Wiegert (1999),
    Table 1, for S-type planets.
    """
    rho_crit = 0.464 - 0.380 * mu - 0.631 * e_bin + 0.150 * mu * e_bin + 0.198 * e_bin**2 + 0.088 * mu * e_bin**2
    return rho_crit

def run_short_simulation_and_extract_features(params):
    """
    Sets up and runs a short rebound simulation for a given set of parameters with TWO planets,
    then extracts the specified features for each planet and planet-planet interactions.

    Args:
        params (dict): A dictionary containing the parameters for one system:
                       'mu', 'e_bin', 'rho_p1', 'e_p1', 'inc_p1', 'mean_anomaly_p1',
                       'rho_p2', 'e_p2', 'inc_p2', 'mean_anomaly_p2', 'id'.

    Returns:
        tuple: (dict: extracted features, str: reason for instability or 'ok'). Returns NaNs for features
              if an escape or error occurs.
    """
    mu = params['mu']
    e_bin = params['e_bin']

    # Planet 1 parameters
    rho_p1 = params['rho_p1']
    e_p1 = params['e_p1']
    inc_p1_rad = params['inc_p1'] # Already in radians
    mean_anomaly_p1_rad = params['mean_anomaly_p1'] # Already in radians

    # Planet 2 parameters
    rho_p2 = params['rho_p2']
    e_p2 = params['e_p2']
    inc_p2_rad = params['inc_p2'] # Already in radians
    mean_anomaly_p2_rad = params['mean_anomaly_p2'] # Already in radians

    system_id = params.get('id', 'N/A')

    # Calculate stellar masses
    m_B = mu * M_total
    m_A = M_total - m_B

    # Planetary semimajor axes
    a_p1 = rho_p1 * a_bin
    a_p2 = rho_p2 * a_bin

    sim = rebound.Simulation()
    sim.integrator = "ias15"
    sim.G = 1.0

    # Add primary star (m_A) at origin
    sim.add(m=m_A)

    # Add secondary star (m_B) orbiting primary
    sim.add(m=m_B, primary=sim.particles[0], a=a_bin, e=e_bin, inc=0, Omega=0, omega=0, M=0)

    # Add Planet 1 (test particle) orbiting primary
    # M_p_ref_for_features is a small mass to enable Hill radius calculations, etc.
    sim.add(m=M_p_ref_for_features, primary=sim.particles[0], a=a_p1, e=e_p1, inc=inc_p1_rad, Omega=0, omega=0, M=mean_anomaly_p1_rad)

    # Add Planet 2 (test particle) orbiting primary
    sim.add(m=M_p_ref_for_features, primary=sim.particles[0], a=a_p2, e=e_p2, inc=inc_p2_rad, Omega=0, omega=0, M=mean_anomaly_p2_rad)

    # Move the entire system to its center of momentum frame
    sim.move_to_com()

    # Set initial timestep for IAS15
    sim.dt = sim.particles[1].P / 20.

    # Initialize MEGNO
    sim.init_megno()

    # Store orbital elements and MEGNO values for both planets
    times = np.linspace(0, T_short, N_output_points)

    # Planet 1 data
    megnos_p1 = []
    a_ps_p1 = []
    e_vec_x_p1 = []
    e_vec_y_p1 = []
    inc_ps_p1 = [] # To track inclination for mutual inclination

    # Planet 2 data
    megnos_p2 = []
    a_ps_p2 = []
    e_vec_x_p2 = []
    e_vec_y_p2 = []
    inc_ps_p2 = [] # To track inclination for mutual inclination

    # Initial features dictionary with default NaN values
    features = {
        'mu': mu,
        'e_bin': e_bin,
        'rho_crit_HW99_p1': calculate_rho_crit_HW99(mu, e_bin),

        # Planet 1 features
        'rho_p1': rho_p1,
        'e_p1': e_p1,
        'inc_p_deg_p1': np.rad2deg(inc_p1_rad),
        'megno_median_p1': np.nan,
        'megno_std_p1': np.nan,
        'e_p_free_p1': np.nan,
        'e_p_forced_p1': np.nan,
        'a_p_std_p1': np.nan,

        # Planet 2 features
        'rho_p2': rho_p2,
        'e_p2': e_p2,
        'inc_p_deg_p2': np.rad2deg(inc_p2_rad),
        'megno_median_p2': np.nan,
        'megno_std_p2': np.nan,
        'e_p_free_p2': np.nan,
        'e_p_forced_p2': np.nan,
        'a_p_std_p2': np.nan,

        # Planet-planet interaction features
        'period_ratio_p2_p1': np.nan,
        'initial_separation_au': np.nan,
        'init_mutual_inc_deg': np.nan,
        'mmr_flag_2_1': 0,
        'mmr_flag_3_2': 0,
        'stable': 0 # Default to unstable, will be set to 1 later if stable
    }
    stability_reason = 'ok'

    try:
        for i, time in enumerate(times):
            sim.integrate(time)

            # Planet 1 (index 2)
            megnos_p1.append(sim.megno(particle=sim.particles[2]))
            a_ps_p1.append(sim.particles[2].a)
            e_vec_x_p1.append(sim.particles[2].e * np.cos(sim.particles[2].pomega))
            e_vec_y_p1.append(sim.particles[2].e * np.sin(sim.particles[2].pomega))
            inc_ps_p1.append(sim.particles[2].inc)

            # Planet 2 (index 3)
            megnos_p2.append(sim.megno(particle=sim.particles[3]))
            a_ps_p2.append(sim.particles[3].a)
            e_vec_x_p2.append(sim.particles[3].e * np.cos(sim.particles[3].pomega))
            e_vec_y_p2.append(sim.particles[3].e * np.sin(sim.particles[3].pomega))
            inc_ps_p2.append(sim.particles[3].inc)

    except rebound.Escape as error:
        print(f"System {system_id} had an Escape during short integration.")
        stability_reason = 'escape'
        return features, stability_reason
    except rebound.Encounter as error:
        print(f"System {system_id} had an Encounter during short integration.")
        stability_reason = 'encounter'
        return features, stability_reason
    except Exception as e:
        sys.stderr.write(f"System {system_id} had an unexpected error during short integration: {e}\n")
        stability_reason = 'short_error'
        return features, stability_reason

    # Convert lists to numpy arrays for calculations
    megnos_p1 = np.array(megnos_p1)
    a_ps_p1 = np.array(a_ps_p1)
    e_vec_x_p1 = np.array(e_vec_x_p1)
    e_vec_y_p1 = np.array(e_vec_y_p1)
    inc_ps_p1 = np.array(inc_ps_p1)

    megnos_p2 = np.array(megnos_p2)
    a_ps_p2 = np.array(a_ps_p2)
    e_vec_x_p2 = np.array(e_vec_x_p2)
    e_vec_y_p2 = np.array(e_vec_y_p2)
    inc_ps_p2 = np.array(inc_ps_p2)

    # Calculate features for Planet 1
    features['megno_median_p1'] = np.nanmedian(megnos_p1[int(0.9 * N_output_points):]) if len(megnos_p1) > 0 else np.nan
    features['megno_std_p1'] = np.nanstd(megnos_p1[int(0.2 * N_output_points):]) if len(megnos_p1) > 0 else np.nan

    if len(e_vec_x_p1) > 0:
        mean_e_vec_x_p1 = np.nanmean(e_vec_x_p1)
        mean_e_vec_y_p1 = np.nanmean(e_vec_y_p1)
        features['e_p_forced_p1'] = np.sqrt(mean_e_vec_x_p1**2 + mean_e_vec_y_p1**2)
        e_p_free_components_x_p1 = e_vec_x_p1 - mean_e_vec_x_p1
        e_p_free_components_y_p1 = e_vec_y_p1 - mean_e_vec_y_p1
        e_p_free_instantaneous_p1 = np.sqrt(e_p_free_components_x_p1**2 + e_p_free_components_y_p1**2)
        features['e_p_free_p1'] = np.nanmean(e_p_free_instantaneous_p1)
    else:
        features['e_p_forced_p1'] = np.nan
        features['e_p_free_p1'] = np.nan

    features['a_p_std_p1'] = np.nanstd(a_ps_p1) / a_p1 if a_p1 != 0 and len(a_ps_p1) > 0 else np.nan

    # Calculate features for Planet 2
    features['megno_median_p2'] = np.nanmedian(megnos_p2[int(0.9 * N_output_points):]) if len(megnos_p2) > 0 else np.nan
    features['megno_std_p2'] = np.nanstd(megnos_p2[int(0.2 * N_output_points):]) if len(megnos_p2) > 0 else np.nan

    if len(e_vec_x_p2) > 0:
        mean_e_vec_x_p2 = np.nanmean(e_vec_x_p2)
        mean_e_vec_y_p2 = np.nanmean(e_vec_y_p2)
        features['e_p_forced_p2'] = np.sqrt(mean_e_vec_x_p2**2 + mean_e_vec_y_p2**2)
        e_p_free_components_x_p2 = e_vec_x_p2 - mean_e_vec_x_p2
        e_p_free_components_y_p2 = e_vec_y_p2 - mean_e_vec_y_p2
        e_p_free_instantaneous_p2 = np.sqrt(e_p_free_components_x_p2**2 + e_p_free_components_y_p2**2)
        features['e_p_free_p2'] = np.nanmean(e_p_free_instantaneous_p2)
    else:
        features['e_p_forced_p2'] = np.nan
        features['e_p_free_p2'] = np.nan

    features['a_p_std_p2'] = np.nanstd(a_ps_p2) / a_p2 if a_p2 != 0 and len(a_ps_p2) > 0 else np.nan

    # Calculate Planet-Planet Interaction Features
    if sim.particles[2].P > 0 and sim.particles[3].P > 0:
        features['period_ratio_p2_p1'] = sim.particles[3].P / sim.particles[2].P
    else:
        features['period_ratio_p2_p1'] = np.nan

    features['initial_separation_au'] = a_p2 - a_p1 # Assumes a_p2 > a_p1

    # Mutual inclination (assuming initial Omega=0 for both)
    features['init_mutual_inc_deg'] = np.rad2deg(np.abs(inc_p1_rad - inc_p2_rad))

    # Mean motion resonance flags (initial state)
    if not np.isnan(features['period_ratio_p2_p1']):
        if 1.95 <= features['period_ratio_p2_p1'] <= 2.05:
            features['mmr_flag_2_1'] = 1
        if 1.45 <= features['period_ratio_p2_p1'] <= 1.55:
            features['mmr_flag_3_2'] = 1

    features['stable'] = -1 # Placeholder, will be updated by long integration if short integration was successful

    return features, stability_reason

def run_long_simulation_and_label_stability(initial_params):
    """
    Sets up and runs a long rebound simulation for a given set of parameters
    with TWO planets and determines the stability label.

    Args:
        initial_params (dict): A dictionary containing the initial parameters for the system.
                                Expected to contain 'mu', 'e_bin', 'rho_p1', 'e_p1', 'inc_p1',
                                'rho_p2', 'e_p2', 'inc_p2', 'id'.

    Returns:
        tuple: (int: 1 for stable, 0 for unstable, str: reason for instability or 'stable').
    """
    mu = initial_params['mu']
    e_bin = initial_params['e_bin']

    # Planet 1 parameters
    rho_p1 = initial_params['rho_p1']
    e_p1 = initial_params['e_p1']
    inc_p1_rad = initial_params['inc_p1'] # Already in radians
    mean_anomaly_p1_rad = initial_params['mean_anomaly_p1'] # Already in radians

    # Planet 2 parameters
    rho_p2 = initial_params['rho_p2']
    e_p2 = initial_params['e_p2']
    inc_p2_rad = initial_params['inc_p2'] # Already in radians
    mean_anomaly_p2_rad = initial_params['mean_anomaly_p2'] # Already in radians

    system_id = initial_params.get('id', 'N/A')

    m_B = mu * M_total
    m_A = M_total - m_B
    a_p1 = rho_p1 * a_bin
    a_p2 = rho_p2 * a_bin

    sim = rebound.Simulation()
    sim.integrator = "ias15"
    sim.G = 1.0

    sim.add(m=m_A)
    sim.add(m=m_B, primary=sim.particles[0], a=a_bin, e=e_bin, inc=0, Omega=0, omega=0, M=0)
    sim.add(m=M_p_ref_for_features, primary=sim.particles[0], a=a_p1, e=e_p1, inc=inc_p1_rad, Omega=0, omega=0, M=mean_anomaly_p1_rad)
    sim.add(m=M_p_ref_for_features, primary=sim.particles[0], a=a_p2, e=e_p2, inc=inc_p2_rad, Omega=0, omega=0, M=mean_anomaly_p2_rad)

    sim.move_to_com()
    sim.dt = sim.particles[1].P / 20.

    # Set up exit conditions for instability for both planets
    # Check if either planet gets too close to any other body
    # This rebound.Simulation.exit_min_distance applies to all particles relative to any other
    sim.exit_min_distance = MIN_DIST_FACTOR * a_bin

    stability_label = 0
    stability_reason = 'long_error'

    try:
        sim.integrate(T_long)
        stability_label = 1
        stability_reason = 'stable'
    except rebound.Escape as error:
        print(f"System {system_id} had an Escape during long integration.")
        stability_reason = 'escape'
    except rebound.Encounter as error:
        print(f"System {system_id} had an Encounter during long integration.")
        stability_reason = 'encounter'
    except Exception as e:
        sys.stderr.write(f"System {system_id} had an unexpected error during long integration: {e}\n")
        stability_reason = 'long_error'

    return stability_label, stability_reason

def worker_simulate_system(system_params):
    """
    Worker function to run both short and long simulations for a single system.
    """
    # First, run short simulation and extract features
    features, short_reason = run_short_simulation_and_extract_features(system_params)

    # If the short simulation already indicated instability (e.g., due to escape/encounter/error),
    # don't run long integration and mark as unstable with the reason from the short sim.
    if short_reason != 'ok':
        features['stable'] = 0
        features['reason'] = short_reason
    else:
        # Then, run long simulation to determine stability label
        stability_label, long_reason = run_long_simulation_and_label_stability(system_params)
        features['stable'] = stability_label
        features['reason'] = long_reason

    return features

def main():
    start_time = time.time() # Record start time
    N_SYSTEMS = int(os.getenv('N_SYSTEMS', '100')) # Default to 100 for quick testing
    OUTPUT_FILENAME = f"s_type_stability_data_two_planets_{N_SYSTEMS}.csv" # Updated filename
    CHECKPOINT_INTERVAL = 50
    N_WORKERS = int(os.getenv('N_WORKERS', '11')) # Default to 11 workers

    print(f"Generating {N_SYSTEMS} two-planet systems using {N_WORKERS} workers.")
    print(f"Output will be saved to {OUTPUT_FILENAME} with checkpointing every {CHECKPOINT_INTERVAL} systems.")

    list_of_params_for_workers = []
    num_sampled = 0
    while num_sampled < N_SYSTEMS:
        # Sample parameters for the binary
        mu = np.random.uniform(*mu_range)
        e_bin = np.random.uniform(*e_bin_range)

        # Sample parameters for Planet 1
        rho_p1 = np.random.uniform(rho_min_global, rho_max_global)
        e_p1 = np.random.uniform(*e_p_range)
        inc_p1 = np.random.uniform(*inc_p_range_rad)
        mean_anomaly_p1 = np.random.uniform(*mean_anomaly_range_rad)

        # Sample parameters for Planet 2 (must be outside Planet 1 and ensure separation)
        rho_p2 = np.random.uniform(rho_p1 + (0.01 * a_bin), rho_max_global) # Ensure a_p2 > a_p1
        e_p2 = np.random.uniform(*e_p_range)
        inc_p2 = np.random.uniform(*inc_p_range_rad)
        mean_anomaly_p2 = np.random.uniform(*mean_anomaly_range_rad)

        # Check period ratio to avoid very close initial conditions
        if rho_p1 > 0 and rho_p2 > 0: # Ensure no division by zero
            P1 = 2 * np.pi * np.sqrt( (rho_p1 * a_bin)**3 / M_total) # Approx period for planet 1
            P2 = 2 * np.pi * np.sqrt( (rho_p2 * a_bin)**3 / M_total) # Approx period for planet 2

            if P2 / P1 < min_period_ratio_between_planets:
                continue # Resample if too close
        else:
            continue # Resample if rho is non-positive

        list_of_params_for_workers.append({
            'id': num_sampled,
            'mu': mu,
            'e_bin': e_bin,
            'rho_p1': rho_p1,
            'e_p1': e_p1,
            'inc_p1': inc_p1,
            'mean_anomaly_p1': mean_anomaly_p1,
            'rho_p2': rho_p2,
            'e_p2': e_p2,
            'inc_p2': inc_p2,
            'mean_anomaly_p2': mean_anomaly_p2
        })
        num_sampled += 1

    # Prepare CSV file header if it doesn't exist
    # We need to run one dummy simulation to get all feature keys for the header
    dummy_features = worker_simulate_system(list_of_params_for_workers[0])[0]
    if 'id' in dummy_features:
        del dummy_features['id']
    if 'reason' in dummy_features:
        del dummy_features['reason']
    fieldnames = list(dummy_features.keys())

    file_exists = os.path.isfile(OUTPUT_FILENAME)
    if not file_exists:
        with open(OUTPUT_FILENAME, 'w', newline='') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()

    # Counters for final metrics
    escape_count = 0
    encounter_count = 0
    stable_count = 0
    other_unstable_count = 0

    # Use multiprocessing Pool
    completed_systems = 0
    with multiprocessing.Pool(processes=N_WORKERS) as pool:
        for i, result_features in enumerate(tqdm(
            pool.imap_unordered(worker_simulate_system, list_of_params_for_workers),
            total=N_SYSTEMS, desc="Simulating systems")
        ):
            if result_features['stable'] == 1:
                stable_count += 1
            elif result_features['reason'] == 'escape':
                escape_count += 1
            elif result_features['reason'] == 'encounter':
                encounter_count += 1
            else:
                other_unstable_count += 1

            if 'id' in result_features:
                del result_features['id']
            if 'reason' in result_features:
                del result_features['reason']

            with open(OUTPUT_FILENAME, 'a', newline='') as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writerow(result_features)

            completed_systems += 1
            if completed_systems % CHECKPOINT_INTERVAL == 0:
                sys.stdout.write(f"\nCheckpoint: {completed_systems}/{N_SYSTEMS} systems processed and saved.\n")

    end_time = time.time()
    elapsed_time = end_time - start_time

    print(f"\nFinished generating {N_SYSTEMS} systems. Data saved to {OUTPUT_FILENAME}.")
    print(f"\nSummary of results:")
    print(f"  {escape_count} systems had an Escape.")
    print(f"  {encounter_count} systems had an Encounter.")
    print(f"  {stable_count} systems were stable.")
    print(f"  {other_unstable_count} systems were unstable due to other errors (short/long integration). ")
    print(f"Total elapsed time: {elapsed_time:.2f} seconds.")

    if 'ipykernel' in sys.modules:
        df_final = pd.read_csv(OUTPUT_FILENAME)
        print("\nFinal DataFrame head:")
        display(df_final.head())
        print(f"Total stable systems: {df_final['stable'].sum()} out of {len(df_final)}")
        print(f"Total unstable systems: {len(df_final) - df_final['stable'].sum()} out of {len(df_final)}")


if __name__ == '__main__':
    main()